<a href="https://colab.research.google.com/github/asomarufaso/-/blob/main/%E2%94%81%E2%94%81%E2%98%86_%E0%A6%AC%E0%A6%BE%E0%A6%A1%E0%A6%BC%E0%A6%BF_%E0%A6%AD%E0%A6%BE%E0%A6%A1%E0%A6%BC%E0%A6%BE%E0%A6%B0_%E0%A6%AC%E0%A6%BF%E0%A6%AC%E0%A6%B0%E0%A6%A3_%E2%98%86%E2%94%81%E2%94%81_%F0%9F%8C%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# প্রথমে প্রয়োজনীয় টুলস ইনস্টল করে নেওয়া হচ্ছে
!pip install -q gradio pandas openpyxl

import gradio as gr
import pandas as pd
import io

# ইংলিশ সংখ্যা ও অক্ষরকে বাংলায় রূপান্তর করার ফাংশন
def to_bangla(text):
    if text is None:
        return ""
    text = str(text)

    # সংখ্যার ম্যাপিং
    numbers = {'0': '০', '1': '১', '2': '২', '3': '৩', '4': '৪', '5': '৫', '6': '৬', '7': '৭', '8': '৮', '9': '৯'}
    # ফ্ল্যাট অক্ষরের ম্যাপিং (সাধারণ কিছু উদাহরণ)
    alphabets = {
        'A': 'এ', 'B': 'বি', 'C': 'সি', 'D': 'ডি', 'E': 'ই', 'F': 'এফ',
        'a': 'এ', 'b': 'বি', 'c': 'সি', 'd': 'ডি', 'e': 'ই', 'f': 'এফ'
    }

    # প্রথমে অক্ষর পরিবর্তন
    for eng, ban in alphabets.items():
        text = text.replace(eng, ban)
    # তারপর সংখ্যা পরিবর্তন
    for eng, ban in numbers.items():
        text = text.replace(eng, ban)

    return text

# মাসের নাম বাংলায় রূপান্তর (যদি কেউ ইংলিশে লেখে)
def month_to_bangla(month_text):
    months = {
        'January': 'জানুয়ারী', 'February': 'ফেব্রুয়ারী', 'March': 'মার্চ', 'April': 'এপ্রিল',
        'May': 'মে', 'June': 'জুন', 'July': 'জুলাই', 'August': 'আগস্ট',
        'September': 'সেপ্টেম্বর', 'October': 'অক্টোবর', 'November': 'নভেম্বর', 'December': 'ডিসেম্বর'
    }
    for eng, ban in months.items():
        if eng.lower() in month_text.lower():
            # বছর আলাদা করে সংখ্যা বাংলায় নেওয়া
            year = "".join([c for c in month_text if c.isdigit()])
            if year:
                return f"{ban} {to_bangla(year)}"
            return ban
    return to_bangla(month_text)

# মূল রিসিট জেনারেটর ফাংশন
def generate_receipt_text(flat_no, month_year, rent_amount, unit_consumed, per_unit_cost, water_bill, cleaning_bill, arrears, advance):
    try:
        # হিসাবের জন্য সংখ্যায় রূপান্তর
        rent = int(rent_amount)
        unit = float(unit_consumed)
        rate = float(per_unit_cost)
        water = int(water_bill)
        cleaning = int(cleaning_bill)
        arr = int(arrears)
        adv = int(advance)

        # গাণিতিক হিসাব
        electricity_total = round(unit * rate)
        total_bill = rent + electricity_total + water + cleaning + arr
        net_payable = total_bill - adv

        # বাংলায় রূপান্তরকরণ
        flat_bn = to_bangla(flat_no)
        month_bn = month_to_bangla(month_year)
        rent_bn = to_bangla(f"{rent:,}")
        unit_bn = to_bangla(str(unit_consumed))
        rate_bn = to_bangla(str(per_unit_cost))
        elec_bn = to_bangla(f"{electricity_total:,}")
        water_bn = to_bangla(f"{water:,}")
        cleaning_bn = to_bangla(f"{cleaning:,}")
        arr_bn = to_bangla(f"{arr:,}")
        adv_bn = to_bangla(f"{adv:,}")
        net_bn = to_bangla(f"{net_payable:,}")

        # ফরম্যাট তৈরি
        return f"""🌟 ━━☆ বাড়ি ভাড়ার বিবরণ ☆━━ 🌟
🏡 ফ্ল্যাট: {flat_bn}
📅 মাস: {month_bn}
📋 বিস্তারিত হিসাব:
━━━━━━━━━━━━━━━━━━━
🏠 বাড়ি ভাড়া: {rent_bn}৳
⚡ বৈদ্যুতিক বিল: {unit_bn}×{rate_bn} = {elec_bn}৳
🚿 পানির বিল: {water_bn}৳
🧹🗑 ক্লিনিং ও আবর্জনা বিল: {cleaning_bn}৳
📌 বকেয়া: {arr_bn}৳
💰 অগ্রিম জমা: -{adv_bn}৳
━━━━━━━━━━━━━━━━━━━
💵 মোট পরিশোধযোগ্য: ({rent_bn}+{elec_bn}+{water_bn}+{cleaning_bn}+{arr_bn} - {adv_bn})
🎯 {net_bn} ৳ ✅

🌟 শুভেচ্ছান্তে,
💁‍♂️ মারুফ
"""
    except Exception as e:
        return f"ভুল ইনপুট! দয়া করে নিশ্চিত করুন সব ঘরের তথ্য ঠিক আছে কিনা।"

# এক্সেল ফাইলের জন্য বাল্ক প্রসেস
def bulk_process(file):
    if file is None:
        return "দয়া করে একটি এক্সেল ফাইল আপলোড করুন।"
    try:
        df = pd.read_excel(file.name)
        all_receipts = ""
        for index, row in df.iterrows():
            receipt = generate_receipt_text(
                row['flat_no'], row['month_year'], row['rent_amount'],
                row['unit_consumed'], row['per_unit_cost'], row['water_bill'],
                row['cleaning_bill'], row['arrears'], row['advance']
            )
            all_receipts += receipt + "\n" + "="*40 + "\n\n"
        return all_receipts
    except Exception as e:
        return "ফাইল প্রসেস করতে সমস্যা হয়েছে। কলামের নামগুলো ঠিক আছে কিনা চেক করুন।"

# গ্রাডিও ইন্টারফেস (অনলাইন ওয়েব অ্যাপ ডিজাইন)
with gr.Blocks(title="Maruf Rent Generator") as demo:
    gr.Markdown("# 🏠 মারুফ ভাইয়ের বাড়ি ভাড়া জেনারেটর (Auto-Bangla Converter)")

    with gr.Tab("সিঙ্গেল ইনপুট (Manual)"):
        gr.Markdown("👉 **এখানে আপনি ইংলিশে (যেমন: 1A, 5000, January 2026) লিখলেও আউটপুট স্বয়ংক্রিয়ভাবে বাংলায় তৈরি হবে।**")
        with gr.Row():
            with gr.Column():
                flat = gr.Textbox(label="ফ্ল্যাট নম্বর (যেমন: 1A বা 2B)", value="1A")
                month = gr.Textbox(label="মাস ও বছর (যেমন: January 2026 বা জানুয়ারী 2026)", value="January 2026")
                rent = gr.Number(label="বাড়ি ভাড়া", value=5000)
                unit = gr.Number(label="বিদ্যুৎ ইউনিট", value=9)
                rate = gr.Number(label="প্রতি ইউনিট রেট", value=102.77)
                water = gr.Number(label="পানির বিল", value=100)
                cleaning = gr.Number(label="ক্লিনিং বিল", value=160)
                arrears = gr.Number(label="বকেয়া (না থাকলে 0)", value=930)
                advance = gr.Number(label="অগ্রিম জমা (না থাকলে 0)", value=0)
                btn1 = gr.Button("হিসাব তৈরি করুন 🚀", variant="primary")
            with gr.Column():
                output1 = gr.Textbox(label="হোয়াটসঅ্যাপ মেসেজ (কপি করার জন্য)", lines=18)

        btn1.click(generate_receipt_text, inputs=[flat, month, rent, unit, rate, water, cleaning, arrears, advance], outputs=output1)

    with gr.Tab("এক্সেল আপলোড (Bulk)"):
        gr.Markdown("👉 **এক্সেলে সব ডাটা ইংলিশে থাকলেও সমস্যা নেই, সব অটোমেটিক বাংলা হয়ে যাবে।**")
        file_input = gr.File(label="আপনার এক্সেল ফাইলটি আপলোড করুন (.xlsx)")
        btn2 = gr.Button("সবার হিসাব একসাথে তৈরি করুন 🚀", variant="primary")
        output2 = gr.Textbox(label="সব ফ্ল্যাটের মেসেজ একসাথে", lines=20)

        btn2.click(bulk_process, inputs=file_input, outputs=output2)

# রান করা এবং পাবলিক লিংক জেনারেট করা
demo.launch(share=True)